In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
from src.utils.dataframe_utils import create_dataframe, merge_dataset

df = create_dataframe("../data/train.csv", "../data/boneage-training-dataset")
df_val = create_dataframe("../data/validation.csv", "../data/boneage-validation-dataset")



In [ ]:
from sklearn.preprocessing import StandardScaler

paths = df["path"].values
labels = df[["boneage"]].values.astype("float32")

scaler = StandardScaler()
scaler.fit(labels)
labels_scaled = scaler.transform(labels)

paths_val = df_val["path"].values
labels_val = df_val[["boneage"]].values.astype("float32")
labels_val_scaled = scaler.transform(labels_val)

In [5]:
"""
#load of the segmented data
df_seg = create_dataframe("../data/train.csv", "../data/boneage-training-segmented")
df_val_seg = create_dataframe("../data/validation.csv", "../data/boneage-validation-dataset-segmented")

"""


'\n#load of the segmented data\ndf_seg = create_dataframe("../data/train.csv", "../data/boneage-training-segmented")\ndf_val_seg = create_dataframe("../data/validation.csv", "../data/boneage-validation-dataset-segmented")\n\n'

In [6]:
"""
#create a new dataset with the segmented data

df = merge_dataset(df, df_segmented)
df_val = merge_dataset(df_val, df_val_segmented)
"""

'\n#create a new dataset with the segmented data\n\ndf = merge_dataset(df, df_segmented)\ndf_val = merge_dataset(df_val, df_val_segmented)\n'

In [7]:
from torch.utils.data import Dataset
from PIL import Image

class BoneAgeDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = Image.open(row["path"]).convert("L")

        label = row["boneage"]

        if self.transform:
            image = self.transform(image)

        return image, label

In [8]:
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

train_dataset = BoneAgeDataset(df, transform)
val_dataset = BoneAgeDataset(df_val, transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: cpu


In [4]:
import torch
print(torch.cuda.is_available())
print(torch.version.cuda)

False
None


In [10]:

import torch.nn as nn

class BoneAgeCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x):

        x = self.features(x)

        x = self.regressor(x)

        return x

In [11]:
model = BoneAgeCNN().to(device)

print(model)

BoneAgeCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (regressor): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=128, out_features=64, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


Let's train the CNN net

In [ ]:
import torch.optim as optim

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
import torch
import json


num_epochs = 50

history = {
    "train_loss": [],
    "val_loss": [],
    "train_mae": [],
    "val_mae": []
}


best_val_loss = float("inf")


for epoch in range(num_epochs):

    # =====================
    # TRAIN
    # =====================

    model.train()

    train_loss = 0
    train_mae = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.float().to(device)

        # forward
        outputs = model(images)

        # loss
        loss = criterion(
            outputs.squeeze(),
            labels
        )


        # backward
        optimizer.zero_grad()

        loss.backward()

        optimizer.step()


        train_loss += loss.item()

        mae = torch.mean(
            torch.abs(outputs.squeeze() - labels)
        )

        train_mae += mae.item()



    train_loss /= len(train_loader)
    train_mae /= len(train_loader)



    # =====================
    # VALIDATION
    # =====================

    model.eval()

    val_loss = 0
    val_mae = 0


    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.float().to(device)


            outputs = model(images)


            loss = criterion(
                outputs.squeeze(),
                labels
            )


            val_loss += loss.item()


            mae = torch.mean(
                torch.abs(outputs.squeeze() - labels)
            )

            val_mae += mae.item()



    val_loss /= len(val_loader)
    val_mae /= len(val_loader)



    # salvo history

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_mae"].append(train_mae)
    history["val_mae"].append(val_mae)



    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.2f} "
        f"Val Loss: {val_loss:.2f} "
        f"Train MAE: {train_mae:.2f} "
        f"Val MAE: {val_mae:.2f}"
    )


    # =====================
    # SALVA MIGLIOR MODELLO
    # =====================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            "boneage_cnn_best.pth"
        )



# salvo history

with open("history.json", "w") as f:
    json.dump(history, f)

In [ ]:
model = BoneAgeCNN().to(device)

model.load_state_dict(
    torch.load("boneage_cnn_best.pth")
)

model.eval()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history["train_mae"], label="train")
plt.plot(history["val_mae"], label="val")

plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()
plt.show()